<a href="https://colab.research.google.com/github/lilianeuwase2/math_forML-Formative3/blob/main/ProbabilityDistributions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("GaltonFamilies.csv")

#use drop duplicates on the mothers column so that we have just one mother per family
mothers = df.drop_duplicates('family')['mother'].values
children = df['childHeight'].values

x = np.concatenate([mothers, children])
print(len(mothers), len(children), len(x))

205 934 1139


Initialization (iteration 0)

In [ ]:
mean_x = np.mean(x) # this is the mean for all of the heights
std_x = np.std(x)

# to get the guesses for the mu1 and mu2, we can add or subtract the std
mu1 = mean_x - std_x # this is the mean for group 1 height
mu2 = mean_x + std_x # this is the mean for group 2 height

# we will use the same variance because we do not have the curves yet
var1 = np.var(x)
var2 = np.var(x)
pi1 = 0.5
pi2 = 0.5

print(mu1, mu2, var1, var2, pi1, pi2)

62.70521103694457 69.7989153897455 12.580160361236732 12.580160361236732 0.5 0.5


Expection step


In [ ]:
def gaussian(x, mu, var):
    return (1 / np.sqrt(2 * np.pi * var)) * np.exp(-(x - mu) ** 2 / (2 * var))

score1 = pi1 * gaussian(x, mu1, var1)
score2 = pi2 * gaussian(x, mu2, var2)
total = score1 + score2

r1 = score1 / total   # percentage group 1, for every height
r2 = score2 / total   # percentage group 2, for every height

print(r1[:5])
print(r2[:5])
print((r1 + r2)[:5])

[0.39609883 0.46510515 0.78072461 0.78072461 0.98752205]
[0.60390117 0.53489485 0.21927539 0.21927539 0.01247795]
[1. 1. 1. 1. 1.]


Maximization step

In [ ]:
N = len(x)

pi1 = r1.sum() / N
pi2 = r2.sum() / N

mu1 = np.sum(r1 * x) / r1.sum()
mu2 = np.sum(r2 * x) / r2.sum()

var1 = np.sum(r1 * (x - mu1) ** 2) / r1.sum()
var2 = np.sum(r2 * (x - mu2) ** 2) / r2.sum()

print(mu1, mu2, var1, var2, pi1, pi2)

64.06934160517648 68.52832144944013 6.501537210755486 8.769482432346381 0.5104885681471353 0.4895114318528648


Log-Likelihood

In [ ]:
def log_likelihood(x, mu1, mu2, var1, var2, pi1, pi2):
    total = pi1 * gaussian(x, mu1, var1) + pi2 * gaussian(x, mu2, var2)
    return np.sum(np.log(total))

print(log_likelihood(x, mu1, mu2, var1, var2, pi1, pi2))

-3041.370921990979


Loop

In [ ]:
# reset to the starting guess (iteration 0)
mu1, mu2 = mean_x - std_x, mean_x + std_x
var1 = var2 = np.var(x)
pi1 = pi2 = 0.5

history = [[0, mu1, mu2, var1, var2, pi1, pi2,
            log_likelihood(x, mu1, mu2, var1, var2, pi1, pi2)]]

for it in range(1, 3):
    # E-step
    s1 = pi1 * gaussian(x, mu1, var1)
    s2 = pi2 * gaussian(x, mu2, var2)
    r1 = s1 / (s1 + s2)
    r2 = s2 / (s1 + s2)

    # M-step (mu first, then var using the new mu)
    pi1, pi2 = r1.sum() / N, r2.sum() / N
    mu1 = np.sum(r1 * x) / r1.sum()
    mu2 = np.sum(r2 * x) / r2.sum()
    var1 = np.sum(r1 * (x - mu1) ** 2) / r1.sum()
    var2 = np.sum(r2 * (x - mu2) ** 2) / r2.sum()

    history.append([it, mu1, mu2, var1, var2, pi1, pi2,
                    log_likelihood(x, mu1, mu2, var1, var2, pi1, pi2)])

table = pd.DataFrame(history, columns=["iter", "mu1", "mu2", "var1", "var2", "pi1", "pi2", "log_lik"])
print(table.round(4))

   iter      mu1      mu2     var1     var2     pi1     pi2    log_lik
0     0  62.7052  69.7989  12.5802  12.5802  0.5000  0.5000 -3190.3325
1     1  64.0693  68.5283   6.5015   8.7695  0.5105  0.4895 -3041.3709
2     2  64.0394  68.5861   6.0192   8.8886  0.5133  0.4867 -3039.7574


E-step separate

In [ ]:
def e_step(x, mu1, mu2, var1, var2, pi1, pi2):
    s1 = pi1 * gaussian(x, mu1, var1)
    s2 = pi2 * gaussian(x, mu2, var2)
    total = s1 + s2
    return s1 / total, s2 / total

M-step separate

In [ ]:
def m_step(x, r1, r2):
    N = len(x)
    pi1, pi2 = r1.sum() / N, r2.sum() / N
    mu1 = np.sum(r1 * x) / r1.sum()
    mu2 = np.sum(r2 * x) / r2.sum()
    var1 = np.sum(r1 * (x - mu1) ** 2) / r1.sum()
    var2 = np.sum(r2 * (x - mu2) ** 2) / r2.sum()
    return mu1, mu2, var1, var2, pi1, pi2

the Loop

In [ ]:
def run_em(x, n_iter):
    params = (x.mean() - x.std(), x.mean() + x.std(), x.var(), x.var(), 0.5, 0.5)
    history = [[0, *params, log_likelihood(x, *params)]]

    for it in range(1, n_iter + 1):
        r1, r2 = e_step(x, *params)
        params = m_step(x, r1, r2)
        history.append([it, *params, log_likelihood(x, *params)])

    cols = ["iter", "mu1", "mu2", "var1", "var2", "pi1", "pi2", "log_lik"]
    return pd.DataFrame(history, columns=cols), params

table, params = run_em(x, 2)
print(table.round(4))

   iter      mu1      mu2     var1     var2     pi1     pi2    log_lik
0     0  62.7052  69.7989  12.5802  12.5802  0.5000  0.5000 -3190.3325
1     1  64.0693  68.5283   6.5015   8.7695  0.5105  0.4895 -3041.3709
2     2  64.0394  68.5861   6.0192   8.8886  0.5133  0.4867 -3039.7574
